# Step 6: Model Evaluation, Threshold Tuning & Comparison




### Create a final comparison table across all models and key threshold variants:

Logistic Regression (best imbalance strategy, default threshold)

Random Forest (default threshold)

XGBoost Baseline (default threshold)

XGBoost Tuned (default threshold 0.5)

XGBoost Tuned (F1-optimal threshold)

XGBoost Tuned (Recall >= 0.90 threshold)

For each row report: Precision (Fraud), Recall (Fraud), F1 (Fraud), PR-AUC, Threshold used.

Plot all Precision-Recall curves on a single figure with a legend.

Write a recommendation (4–6 sentences): Which model and threshold would you deploy at Paytm's fraud team? Justify using specific metric values from your table.


In [6]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    precision_recall_curve
)

In [7]:
import pandas as pd

df = pd.read_excel("creditcard.xlsx")

In [8]:
import numpy as np

df["Amount_log"] = np.log1p(df["Amount"])
df["Hour"] = ((df["Time"] % 86400) // 3600).astype(int)

df.drop(["Time", "Amount"], axis=1, inplace=True)

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df[["Amount_log", "Hour"]] = scaler.fit_transform(
    df[["Amount_log", "Hour"]]
)

In [10]:
X = df.drop("Class", axis=1)
y = df["Class"]

print(X.shape)
print(y.shape)

(284807, 30)
(284807,)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (227845, 30)
X_test : (56962, 30)
y_train: (227845,)
y_test : (56962,)


In [12]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    sampling_strategy=0.1,
    random_state=42
)

X_smote, y_smote = smote.fit_resample(X_train, y_train)

print(X_smote.shape)
print(y_smote.shape)

(250196, 30)
(250196,)


In [13]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    sampling_strategy=0.1,
    random_state=42
)

X_smote, y_smote = smote.fit_resample(
    X_train,
    y_train
)

In [15]:
from sklearn.linear_model import LogisticRegression

# Create model
lr_smote = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train model
lr_smote.fit(X_smote, y_smote)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [16]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

# Prediction
lr_pred = lr_smote.predict(X_test)

# Probability
lr_prob = lr_smote.predict_proba(X_test)[:,1]

# Metrics
lr_precision = precision_score(y_test, lr_pred)
lr_recall = recall_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)
lr_prauc = average_precision_score(y_test, lr_prob)

print("Precision :", lr_precision)
print("Recall :", lr_recall)
print("F1 Score :", lr_f1)
print("PR-AUC :", lr_prauc)

Precision : 0.3686440677966102
Recall : 0.8877551020408163
F1 Score : 0.5209580838323353
PR-AUC : 0.7417930746890438


In [17]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    sampling_strategy=0.1,
    random_state=42
)

X_smote, y_smote = smote.fit_resample(
    X_train,
    y_train
)

In [18]:
from sklearn.model_selection import train_test_split

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)